# Structural credit models

**Start here:** This deep dive expands on `07_advanced_quant/correlation_and_credit_models.ipynb`; read the overview first for the common copula, latent-factor, and recovery vocabulary.

Demonstrates `finstack_quant.valuations.models.credit`: `MertonModel` (including CreditGrades calibration), `DynamicRecoverySpec`, `EndogenousHazardSpec`, `ToggleExerciseModel`, and `CreditState`.


## Imports

In [1]:
from finstack_quant.valuations.models.credit import (
    MertonModel,
    DynamicRecoverySpec,
    EndogenousHazardSpec,
    ToggleExerciseModel,
    CreditState,
)
import json

print("Imports OK")


Imports OK


## `MertonModel` (basic)

In [2]:
m = MertonModel(100.0, 0.25, 80.0, 0.05)
print("MertonModel created")
print("default_probability(1y):", round(m.default_probability(1.0), 6))
print("distance_to_default(1y):", round(m.distance_to_default(1.0), 6))


MertonModel created
default_probability(1y): 0.166629
distance_to_default(1y): 0.967574


## `MertonModel.credit_grades` (equity-calibrated)

In [3]:
cg = MertonModel.credit_grades(
    equity_value=50.0,
    equity_vol=0.30,
    total_debt=40.0,
    risk_free_rate=0.05,
    barrier_uncertainty=0.10,
    mean_recovery=0.40,
)
print("CreditGrades MertonModel created")
print("default_probability(1y):", round(cg.default_probability(1.0), 10))


CreditGrades MertonModel created
default_probability(1y): 1.84e-08


## JSON round-trip

In [4]:
j = m.to_json()
m2 = MertonModel.from_json(j)
print("round-trip dp(1) match:", abs(m.default_probability(1.0) - m2.default_probability(1.0)) < 1e-12)
print("json head:", j[:180], "...")


round-trip dp(1) match: True
json head: {
  "asset_value": 100.0,
  "asset_vol": 0.25,
  "debt_barrier": 80.0,
  "risk_free_rate": 0.05,
  "payout_rate": 0.0,
  "barrier_type": "Terminal",
  "dynamics": "GeometricBrownia ...


## `DynamicRecoverySpec`

In [5]:
dr_const = DynamicRecoverySpec.constant(0.40)
print("constant:", dr_const.to_json())

# Note: other dynamic recovery factories exist (market correlated etc.)
# and are used via the same pattern as RecoverySpec in the correlation tier.


constant: {
  "base_recovery": 0.4,
  "base_notional": 1.0,
  "model": "Constant"
}


## `EndogenousHazardSpec`

In [6]:
eh = EndogenousHazardSpec.power_law(0.05, 1.0, 1.2)
print("power_law spec:", eh.to_json())
print("hazard_at_leverage(2.0):", eh.hazard_at_leverage(2.0))

# hazard_after_pik_accrual and hazard_at_leverage are also available as
# instance methods on specs created via power_law (or from_json).


power_law spec: {
  "base_hazard_rate": 0.05,
  "base_leverage": 1.0,
  "leverage_hazard_map": {
    "PowerLaw": {
      "exponent": 1.2
    }
  }
}
hazard_at_leverage(2.0): 0.11486983549970349


## `ToggleExerciseModel` + `CreditState`

In [7]:
cs = CreditState(0.03, 0.40)
print("CreditState:", cs.to_json())

# ToggleExerciseModel supports from_json/to_json with variants
# (Threshold, Stochastic, OptimalExercise). See the .pyi for full shapes.


CreditState: {
  "hazard_rate": 0.03,
  "distance_to_default": 0.4,
  "leverage": 0.0,
  "accreted_notional": 0.0,
  "coupon_due": 0.0,
  "asset_value": null
}


## Takeaways

- `MertonModel` provides risk-neutral `default_probability` and `distance_to_default` for structural (asset-value) credit analysis. Use `credit_grades(...)` when calibrating from observable equity.
- `DynamicRecoverySpec`, `EndogenousHazardSpec`, and `ToggleExerciseModel` supply the building blocks for path-dependent credit features (PIK, toggles, leverage-dependent hazard).
- `CreditState` is a lightweight (PD, LGD) snapshot used by higher-level credit workflows.
- All types support `to_json()` / `from_json()` for round-tripping.